# Bag-of-words baseline on SciJudgeBench


## Pilot 1: `pilot.py`

- **Training data:** `test_extended.jsonl` (a public evaluation split, not the paper's training split), minus every pair that shares an arXiv ID with `test.jsonl` or `test_ood_year.jsonl`.
- **Simple rules:** older paper wins; longer abstract wins; longer title wins. Ties count as 0.5.
- **Classifier:** TF-IDF logistic regression on the difference between paper A and paper B, with and without date and length features. Three values of C are reported, all on the test sets.

In [ ]:
import json, numpy as np, datetime as dt, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
def load(f): return [json.loads(l) for l in open(f)]
test, year, ext = load('test.jsonl'), load('test_ood_year.jsonl'), load('test_extended.jsonl')
def pid(r,s): return r[f'paper_{s}_arxiv_id']
held = {pid(r,s) for d in (test,year) for r in d for s in 'ab'}
train = [r for r in ext if pid(r,'a') not in held and pid(r,'b') not in held]
print(f"train pairs after removing overlap: {len(train)} of {len(ext)}; label balance test A-share={np.mean([r['correct_answer']=='A' for r in test]):.3f}")
def date(r,s): return dt.datetime.fromisoformat(str(r[f'paper_{s}_date'])[:10])
def heur(d, f):
    acc=[]
    for r in d:
        a,b=f(r,'a'),f(r,'b')
        pred = 'A' if a>b else 'B' if b>a else None
        acc.append(0.5 if pred is None else float(pred==r['correct_answer']))
    return 100*np.mean(acc)
older = lambda r,s: -date(r,s).timestamp()
abslen = lambda r,s: len(r[f'paper_{s}_abstract'].split())
titlen = lambda r,s: len(r[f'paper_{s}_title'].split())
for name,d in (('main test',test),('future-year',year)):
    gaps=[abs((date(r,'a')-date(r,'b')).days) for r in d]
    print(f"{name}: n={len(d)}  older-wins={heur(d,older):.1f}  longer-abstract={heur(d,abslen):.1f}  longer-title={heur(d,titlen):.1f}  median date gap={np.median(gaps):.0f} days")
text = lambda r,s: r[f'paper_{s}_title']+' . '+r[f'paper_{s}_abstract']
vec = TfidfVectorizer(min_df=3, max_features=50000, ngram_range=(1,2), sublinear_tf=True).fit([text(r,s) for r in train for s in 'ab'])
def feats(d, flip=False):
    A = vec.transform([text(r,'a') for r in d]); B = vec.transform([text(r,'b') for r in d])
    extra = np.array([[ (date(r,'b')-date(r,'a')).days/365, np.log1p(abslen(r,'a'))-np.log1p(abslen(r,'b')), titlen(r,'a')-titlen(r,'b')] for r in d])
    from scipy.sparse import hstack, csr_matrix
    return hstack([A-B, csr_matrix(extra)]).tocsr()
# symmetrize training: each pair in both orders
X = feats(train); y = np.array([r['correct_answer']=='A' for r in train])
from scipy.sparse import vstack
Xs, ys = vstack([X, -X]), np.concatenate([y, ~y])
for C in (0.3,1,3):
    clf = LogisticRegression(C=C, max_iter=3000).fit(Xs, ys)
    res = []
    for d in (test, year):
        p = clf.predict(feats(d)); res.append(100*np.mean(p == np.array([r['correct_answer']=='A' for r in d])))
    print(f"TF-IDF+date+length logistic (C={C}): main test {res[0]:.1f}  future-year {res[1]:.1f}")
# date-free variant to separate the date cue
def feats_nodate(d):
    from scipy.sparse import hstack, csr_matrix
    A = vec.transform([text(r,'a') for r in d]); B = vec.transform([text(r,'b') for r in d])
    return (A-B).tocsr()
X2 = feats_nodate(train); clf2 = LogisticRegression(C=1, max_iter=3000).fit(vstack([X2,-X2]), ys)
print("TF-IDF words only (no date, no length): main", round(100*np.mean(clf2.predict(feats_nodate(test))==np.array([r['correct_answer']=='A' for r in test])),1),
      " future-year", round(100*np.mean(clf2.predict(feats_nodate(year))==np.array([r['correct_answer']=='A' for r in year])),1))
coef = clf2.coef_[0]; names = np.array(vec.get_feature_names_out())
print("top 'more cited' words:", ', '.join(names[np.argsort(-coef)[:30]]))
print("top 'less cited' words:", ', '.join(names[np.argsort(coef)[:30]]))

train pairs after removing overlap: 7830 of 8830; label balance test A-share=0.500
main test: n=1000  older-wins=49.6  longer-abstract=58.9  longer-title=48.6  median date gap=110 days
future-year: n=904  older-wins=54.8  longer-abstract=57.2  longer-title=43.3  median date gap=10 days
TF-IDF+date+length logistic (C=0.3): main test 72.0  future-year 76.2
TF-IDF+date+length logistic (C=1): main test 72.1  future-year 76.4
TF-IDF+date+length logistic (C=3): main test 72.0  future-year 75.7
TF-IDF words only (no date, no length): main 71.1  future-year 75.4
top 'more cited' words: large, theory, networks, neural, including, language models, here we, conjecture, llms, https, models, language, nonlinear, research, deep, new, gradient, github, at, survey, tasks, large language, categories, 10, variety of, generation, llm, here, significantly, neural networks
top 'less cited' words: the, some, of, propose, for, in this, is, proposed, let, process, we propose, tree, supersymmetry, psi, that ou

## Pilot 2: `pilot2.py`

- **Training data:** the first 1,200,000,001 bytes of the paper's `train.jsonl` (`train_part.jsonl`); the incomplete final line was skipped.
- **Overlap filter:** removes pairs that share an arXiv ID with either test set.
- **Sampling:** shuffled with `random.seed(0)`, then trained on the first 25,000 pairs, the first 100,000 and all of them.
- **Vocabulary:** fitted on both papers of the first min(n, 60,000) training pairs.
- **Classifier:** liblinear, C = 1.

In [ ]:
import json, numpy as np, collections, random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import vstack
def load(f):
    out=[]
    for l in open(f):
        try: out.append(json.loads(l))
        except Exception: pass
    return out
test, year = load('test.jsonl'), load('test_ood_year.jsonl')
tr = load('train_part.jsonl')
print('train categories', collections.Counter(r['paper_a_category'] for r in tr).most_common(), ' test', collections.Counter(r['paper_a_category'] for r in test).most_common())
held = {r[f'paper_{s}_arxiv_id'] for d in (test,year) for r in d for s in 'ab'}
tr = [r for r in tr if r['paper_a_arxiv_id'] not in held and r['paper_b_arxiv_id'] not in held]
random.seed(0); random.shuffle(tr)
text = lambda r,s: r[f'paper_{s}_title']+' . '+r[f'paper_{s}_abstract']
lab = lambda d: np.array([r['correct_answer']=='A' for r in d])
for n in (25000, 100000, len(tr)):
    sub = tr[:n]
    vec = TfidfVectorizer(min_df=3, max_features=200000, ngram_range=(1,2), sublinear_tf=True, dtype=np.float32).fit([text(r,'a') for r in sub[:60000]]+[text(r,'b') for r in sub[:60000]])
    F = lambda d: (vec.transform([text(r,'a') for r in d]) - vec.transform([text(r,'b') for r in d])).tocsr()
    X = F(sub); y = lab(sub)
    clf = LogisticRegression(C=1, max_iter=2000, solver='liblinear').fit(vstack([X,-X]), np.concatenate([y,~y]))
    acc = [100*np.mean(clf.predict(F(d))==lab(d)) for d in (test,year)]
    bycat = collections.defaultdict(list)
    for r,p in zip(test, clf.predict(F(test))): bycat[r['paper_a_category']].append(p==(r['correct_answer']=='A'))
    print(f"n={n}: main {acc[0]:.1f}  future-year {acc[1]:.1f}  by field: " + ', '.join(f"{k} {100*np.mean(v):.1f} ({len(v)})" for k,v in bycat.items()))

train categories [('Physics', 184899), ('Computer Science', 23536), ('Mathematics', 9881), ('Others', 4397)]  test [('Physics', 340), ('Computer Science', 248), ('Others', 208), ('Mathematics', 204)]
n=25000: main 68.5  future-year 75.6  by field: Others 65.9 (208), Physics 70.6 (340), Mathematics 66.2 (204), Computer Science 69.8 (248)
n=100000: main 72.7  future-year 77.0  by field: Others 72.6 (208), Physics 75.0 (340), Mathematics 70.6 (204), Computer Science 71.4 (248)
n=222713: main 74.6  future-year 79.0  by field: Others 74.0 (208), Physics 76.5 (340), Mathematics 75.0 (204), Computer Science 72.2 (248)


## Comparison with the paper (arXiv 2603.14473v3)

| | Main test (n = 1,000) | Future-year (n = 904) |
|---|---|---|
| Qwen3-4B-Instruct, untrained (paper, Tables 3 and 4) | 58.1 | 64.7 |
| SciJudge-Qwen3-4B (paper) | 77.3 | 80.9 |
| Bag-of-words, 222,713 pairs (pilot 2) | 74.6 | 79.0 |

The metrics differ. The paper counts a pair as correct only if the prediction holds when papers A and B are swapped. The pilot scores each pair once, in the given order. Swap-consistent accuracy was not computed for the baseline. See the caveats in `README.md`.